In [7]:
from langchain_core.tools import tool
import json

from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchAPIWrapper
    
@tool   
def search_news(query, time):
    """Search news articles using search engine."""
    search_engine = DuckDuckGoSearchAPIWrapper(time="w")
    return [json.dumps(article) for article in search_engine._ddgs_news(query=query, max_results=10)]

In [8]:
from typing import List

from langchain_core.pydantic_v1 import BaseModel, Field

   
class Article(BaseModel):
    """Information about a related article"""
    date: str = Field(description="The date of the article")
    title: str = Field(description="The title of the article")
    body: str = Field(description="The body of the article")
    url: str = Field(description="The URL of the article")
    image: str = Field(description="The image of the article")
    source: str = Field(description="The source of the article")

class Response(BaseModel):
    """Final response to the question being asked"""
    articles: list[Article] = Field(description="List of articles")
    # answer: str = Field(description="The final answer to respond to the user")
    # sources: List[int] = Field(
    #     description="List of urls that contain answer to the question. Only include a url if it contains relevant information"
    # )

In [9]:
tools = [search_news, Response]

In [10]:
import json

from langchain_core.agents import AgentActionMessageLog, AgentFinish

def parse(output):
    # If no function was invoked, return to user
    if "function_call" not in output.additional_kwargs:
        return AgentFinish(return_values={"output": output.content}, log=output.content)

    # Parse out the function call
    function_call = output.additional_kwargs["function_call"]
    name = function_call["name"]
    inputs = json.loads(function_call["arguments"])

    # If the Response function was invoked, return to the user with the function inputs
    if name == "Response":
        return AgentFinish(return_values=inputs, log=str(function_call))
    # Otherwise, return an agent action
    else:
        return AgentActionMessageLog(
            tool=name, tool_input=inputs, log="", message_log=[output]
        )

In [11]:
from langchain.agents import AgentExecutor
from langchain.agents.format_scratchpad import format_to_openai_function_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm = ChatOpenAI(temperature=0)

llm_with_tools = llm.bind_functions(tools)

agent = (
    {
        "input": lambda x: x["input"],
        # Format agent scratchpad from intermediate steps
        "agent_scratchpad": lambda x: format_to_openai_function_messages(
            x["intermediate_steps"]
        ),
    }
    | prompt
    | llm_with_tools
    | parse
)

agent_executor = AgentExecutor(tools=[search_news], agent=agent, verbose=True)

In [12]:
agent_executor.invoke(
    {"input": "what are the latest news on AI?"},
    return_only_outputs=True,
)



> Entering new AgentExecutor chain...
['{"date": "2024-04-21T08:51:02+00:00", "title": "AI is helping Amazon send fewer small items in comically large boxes", "body": "Amazon\'s AI-powered Package Decision Engine helps cut down on instances of getting a small item in a huge box.", "url": "https://www.msn.com/en-us/news/other/ai-is-helping-amazon-send-fewer-small-items-in-comically-large-boxes/ar-AA1nnGZb", "image": "https://i.insider.com/6621887420b15f0ff4295cc3?width=1200&format=jpeg", "source": "Business Insider on MSN.com"}', '{"date": "2024-04-21T08:59:00+00:00", "title": "4 signs your email was written by AI\\u2014and why it matters", "body": "AI can be a time-saver for those who struggle with emails. But there are telltale signs that you used AI\\u2014and some serious risks, as well.", "url": "https://www.fastcompany.com/91109655/4-signs-your-email-was-written-by-ai-and-why-it-matters", "image": "https://images.fastcompany.com/image/upload/f_auto,q_auto,c_fit/wp-cms-2/2024/04/p

KeyboardInterrupt: 

c:\Users\Cha\st_projects\Capstone\ai-news-curator\.venv\Lib\site-packages\curl_cffi\aio.py:205: UserWarning: Curlm alread closed! quitting from process_data
  warnings.warn("Curlm alread closed! quitting from process_data")
